# Cross-Modal Diagnostic Observability — Stage T2-I

## Independent target expansion registry, official-source acquisition and harmonisation v0.2

This notebook executes the broad post–T2-H data-expansion strategy in one run:

- verifies the T2-H and T3-PF firewall records;
- freezes a 24-row candidate/exclusion registry before acquisition;
- excludes the three locked-blind sentinel targets and aliases;
- preflights official sources, licences, endpoints and grouping;
- automatically downloads only explicitly public, size-capped assets;
- leaves credentialed, agreement-gated and author-request datasets as auditable `HOLD`;
- hashes, safely extracts and inventories acquired files;
- produces a readiness map for later deduplication and frozen source-score execution.

It does **not** compute target AUC, refit RA-CB-AMW-DDET, access blind data or authorise Stage 12.

Use a clean Colab CPU runtime and select **Runtime → Run all**.


In [1]:
# @title T2-I-0. Mount Drive, verify immutable parents and freeze the expansion registry
import csv, hashlib, io, json, math, os, re, shutil, sys, tarfile, time, warnings, zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except Exception:
    display=print

IN_COLAB=False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB=True
except Exception:
    pass

DEFAULT_ROOT=Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability") if IN_COLAB else Path.cwd()
PROJECT_ROOT=Path(os.environ.get("CDO_PROJECT_ROOT",str(DEFAULT_ROOT)))
CODE_ROOT=PROJECT_ROOT/"05_Code"/"Cross_Modal"
MAP_ROOT=PROJECT_ROOT/"02_Dataset_Map"/"StageT2-I_Independent_Target_Expansion_v0.1"
STUDY_ROOT=PROJECT_ROOT/"04_Study_Design"
ACQ_ROOT=PROJECT_ROOT/"00_Data_Acquisition"/"Cross_Modal_Independent_Target_Expansion_v0.1"
CM_ROOT=PROJECT_ROOT/"06_Data_Records"/"Cross_Modal"
RESULT_ROOT=CM_ROOT/"StageT2-I_Independent_Target_Expansion_Registry_Acquisition_And_Harmonisation_v0.1"
P0,P1,P2,P3,P4=[RESULT_ROOT/x for x in [
    "00_Protocol","01_Access_And_Acquisition","02_Inventory_And_Schema",
    "03_Readiness_And_Gates","04_Results"
]]
for p in [CODE_ROOT,MAP_ROOT,STUDY_ROOT,ACQ_ROOT,P0,P1,P2,P3,P4]:
    p.mkdir(parents=True,exist_ok=True)

NOTEBOOK_NAME="CrossModal_StageT2-I_Independent_Target_Expansion_Registry_Acquisition_And_Harmonisation_v0.2.ipynb"
NOTEBOOK_PATH=CODE_ROOT/NOTEBOOK_NAME
REGISTRY_PATH=MAP_ROOT/"StageT2-I_Independent_Target_Expansion_Candidate_Registry_v0.1.csv"
RATIONALE_PATH=MAP_ROOT/"Independent_Target_Expansion_Rationale_And_Sampling_Frame_v0.1.md"
PREREG_PATH=STUDY_ROOT/"StageT2-I_Independent_Target_Expansion_Registry_Acquisition_And_Harmonisation_Preregistration_v1.0.md"
MANUAL_PATH=STUDY_ROOT/"StageT2-I_Manual_Access_And_Credential_Checklist_v0.1.md"

T2H_FINAL=CM_ROOT/"StageT2-H_Development_Only_Single_Pilot_Deployability_And_Sequential_Forecast_Freeze_v0.1"/"04_Results"/"StageT2-H_Complete_v0.1.json"
T3PF_ACTIVATION=CM_ROOT/"StageT3-PF_Outcome-Free_Preregistration_And_Asset_Preflight_v1.0"/"04_Results"/"StageT3-PF_Activation_Record_v1.0.json"

EXPECTED_SHA={
    "registry":"c6224710027a7419b9b207df35831b5a2671d8900a82b893d19057963edaccd1",
    "rationale":"975e691b6c2a5be82905248d961f3927e1f96c6ebd917f20e983e9d7855d6b5d",
    "prereg":"65df8666ed86eeac2043951400a087d18ba1ed04545fa6c0e30c9e049c89edec",
    "manual":"517ae6c471b5edac05a9316069a2f97bb2cbd6dba90e67244ebfedd37f64b600",
    "t2h_file":"4dc14383a299a97a3937a4fe2a38919952b6c931c5ee14308220141442504da4",
    "t3pf_file":"10646d771a3cd9e86c8c96eb4a134d4878c7542bb4b0b07ab9e01fa8b0c09c25",
}
EXPECTED_T2H_RECORD="27d4c7afe711ba66ea44d11f3ef173820e11ef1eba7a44530446a3e5444aa99f"
EXPECTED_T3PF_RECORD="4397cee7798f684159ed77aa5e1edd7b7ae0a24378047d6c89b37ef9ef738a52"
LOCKED_BLIND_NAMES={
    "BUSI_CAIRO_2019","BUSI_CAIRO","BUSI_2019","OASBUD_2017","OASBUD",
    "DERM7PT_2019","DERM7PT","DERM_7PT"
}

AUTO_DOWNLOAD_OPEN_ASSETS=True
AUTO_EXTRACT_ARCHIVES=True
MAX_TOTAL_AUTO_DOWNLOAD_GB=6.0
MAX_SINGLE_AUTO_DOWNLOAD_GB=3.0
MAX_AUTO_EXTRACT_GB=2.5
REQUEST_TIMEOUT=(20,180)
USER_AGENT="CMDO-StageT2-I/0.1 research-acquisition audit"

def now(): return datetime.now(timezone.utc).isoformat()
def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
def sha_json(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(",",":"),ensure_ascii=False).encode()).hexdigest()
def canonical_csv(frame):
    return frame.fillna("").to_csv(index=False,lineterminator="\n",float_format="%.12g")
def write_once(path,text):
    path=Path(path)
    if path.exists():
        assert path.read_text(encoding="utf-8")==text,f"Replay conflict: {path}"
    else: path.write_text(text,encoding="utf-8")
def write_csv_once(path,frame): write_once(path,canonical_csv(frame))
def write_json_once(path,value): write_once(path,json.dumps(value,indent=2,ensure_ascii=False)+"\n")
def verify_self(path,field):
    value=json.loads(Path(path).read_text(encoding="utf-8"))
    claim=value[field]; core=dict(value); core.pop(field)
    assert sha_json(core)==claim,f"Self-hash mismatch: {path}"
    return value
def notebook_source_sha(path):
    value=json.loads(Path(path).read_text(encoding="utf-8")); cells=[]
    for cell in value.get("cells",[]):
        if cell.get("cell_type") not in {"code","markdown"}: continue
        source=cell.get("source",[]); source="".join(source) if isinstance(source,list) else str(source)
        cells.append({"cell_type":cell["cell_type"],"source":source.replace("\r\n","\n")})
    return sha_json(cells)
def seal(path,payload,field="seal_sha256"):
    path=Path(path)
    if path.exists():
        old=verify_self(path,field)
        for k,v in payload.items(): assert old[k]==v,f"Sealed field changed: {k}"
        return old
    value=dict(payload); value["sealed_utc"]=now(); value[field]=sha_json(value)
    write_json_once(path,value); return value

required=[NOTEBOOK_PATH,REGISTRY_PATH,RATIONALE_PATH,PREREG_PATH,MANUAL_PATH,T2H_FINAL,T3PF_ACTIVATION]
missing=[str(p) for p in required if not p.is_file()]
assert not missing,"Missing required files:\n"+"\n".join(missing)

for role,path in {
    "registry":REGISTRY_PATH,"rationale":RATIONALE_PATH,"prereg":PREREG_PATH,
    "manual":MANUAL_PATH,"t2h_file":T2H_FINAL,"t3pf_file":T3PF_ACTIVATION
}.items():
    assert sha_file(path)==EXPECTED_SHA[role],f"Hash mismatch: {role}"

t2h=verify_self(T2H_FINAL,"final_record_sha256")
assert t2h["final_record_sha256"]==EXPECTED_T2H_RECORD
assert t2h["single_pilot_deployment_authorised"] is False
assert t2h["blind_assets_touched"] is False and t2h["blind_outcomes_accessed"] is False
assert t2h["stage12_authorised"] is False

t3pf=verify_self(T3PF_ACTIVATION,"activation_record_sha256")
assert t3pf["activation_record_sha256"]==EXPECTED_T3PF_RECORD
assert t3pf["blind_assets_acquired"] is False
assert t3pf["blind_outcomes_accessed"] is False
assert t3pf["stage12_authorised"] is False

registry=pd.read_csv(REGISTRY_PATH)
assert len(registry)==24
assert registry["dataset_id"].is_unique
upper_ids={str(x).upper() for x in registry["dataset_id"]}
assert LOCKED_BLIND_NAMES.isdisjoint(upper_ids)
assert not registry.astype(str).apply(lambda c:c.str.upper().isin(LOCKED_BLIND_NAMES)).any().any()
assert (registry["expansion_role"]=="EXCLUDE_DUPLICATE").sum()>=1

protocol=seal(P0/"StageT2-I_Protocol_Seal_v0.1.json",{
    "stage":"StageT2-I",
    "purpose":"independent_target_expansion_registry_acquisition_and_harmonisation",
    "parent_t2h_record":EXPECTED_T2H_RECORD,
    "parent_t3pf_record":EXPECTED_T3PF_RECORD,
    "registry_sha256":EXPECTED_SHA["registry"],
    "rationale_sha256":EXPECTED_SHA["rationale"],
    "preregistration_sha256":EXPECTED_SHA["prereg"],
    "manual_checklist_sha256":EXPECTED_SHA["manual"],
    "notebook_source_sha256":notebook_source_sha(NOTEBOOK_PATH),
    "candidate_rows":int(len(registry)),
    "locked_blind_names_excluded":True,
    "target_outcomes_scored":False,
    "blind_assets_touched":False,
    "blind_outcomes_accessed":False,
    "stage12_authorised":False,
},"protocol_seal_sha256")

print("Stage T2-I protocol:",protocol["protocol_seal_sha256"])
print("Registry rows:",len(registry))
print("Source-axis-compatible rows:",int(registry["source_axis_compatible"].astype(bool).sum()))
print("Blind assets touched:",False)


Mounted at /content/drive
Stage T2-I protocol: 6dcb89a257d0cc45cb80cf8a82094d35a83eb359eba80e196373955b7a88b6b7
Registry rows: 24
Source-axis-compatible rows: 21
Blind assets touched: False


In [2]:
# @title T2-I-1. Official-source, licence, endpoint and grouping preflight
session=requests.Session()
session.headers.update({"User-Agent":USER_AGENT})

def preflight_url(url):
    result={"url":url,"http_status":None,"reachable":False,"final_url":"","content_type":"","content_length":None,"error":""}
    try:
        response=session.get(url,timeout=REQUEST_TIMEOUT,allow_redirects=True,stream=True,headers={"Range":"bytes=0-1023"})
        result.update({
            "http_status":int(response.status_code),
            "reachable":response.status_code in {200,206,301,302,303,307,308,401,403,405},
            "final_url":str(response.url),
            "content_type":response.headers.get("content-type",""),
            "content_length":int(response.headers.get("content-length","0") or 0) or None,
        })
        response.close()
    except Exception as exc:
        result["error"]=f"{type(exc).__name__}: {exc}"
    return result

preflight_rows=[]
for _,row in registry.iterrows():
    result=preflight_url(str(row["official_url"]))
    result.update({
        "dataset_id":row["dataset_id"],
        "modality":row["modality"],
        "expansion_role":row["expansion_role"],
        "official_url":row["official_url"],
        "access_mode":row["access_mode"],
        "licence":row["licence"],
        "grouping_status":row["grouping_status"],
        "label_status":row["label_status"],
        "source_axis_compatible":bool(row["source_axis_compatible"]),
    })
    preflight_rows.append(result)
preflight=pd.DataFrame(preflight_rows)
required_preflight_columns={"official_url","licence","access_mode"}
missing_preflight_columns=required_preflight_columns-set(preflight.columns)
assert not missing_preflight_columns, f"Preflight construction missing columns: {sorted(missing_preflight_columns)}"

# A 401/403 on an explicitly credentialed route is a successful governance preflight, not acquisition.
preflight["governance_route_documented"]=(
    preflight["official_url"].astype(str).str.startswith("https://")
    & preflight["licence"].astype(str).str.len().gt(2)
    & preflight["access_mode"].astype(str).str.len().gt(2)
)

write_csv_once(P1/"StageT2-I_Official_Source_Preflight_v0.1.csv",preflight)
display(preflight[[
    "dataset_id","access_mode","http_status","reachable",
    "governance_route_documented","grouping_status"
]])


,dataset_id,access_mode,http_status,reachable,governance_route_documented,grouping_status
0,MESSIDOR2,MANUAL_OFFICIAL_AGREEMENT,206,True,True,EXAM_LEVEL_AVAILABLE
1,MESSIDOR_ORIGINAL,MANUAL_OFFICIAL_AGREEMENT,206,True,True,EXAM_LEVEL_AVAILABLE
2,BRSET_V1_0_1,CREDENTIALED_DUA,200,True,True,PATIENT_ID_AVAILABLE
3,mBRSET_V1_0,CREDENTIALED_DUA,200,True,True,PATIENT_ID_AVAILABLE
4,ODIR5K_DR,GRAND_CHALLENGE_REGISTRATION,200,True,True,PATIENT_ID_AND_BILATERAL_IMAGES
5,FGADR_SEG,SIGNED_NONCOMMERCIAL_RESEARCH_AGREEMENT,206,True,True,VERIFY_PATIENT_MAPPING
6,DDR,ROUTE_VERIFY,206,True,True,VERIFY_PATIENT_MAPPING
7,ISIC_MILK10K,AUTO_DIRECT_OFFICIAL,200,True,True,LESION_METADATA_AVAILABLE
8,PH2,QUICK_REGISTRATION_MANUAL,206,True,True,LESION_LEVEL
9,ISIC2019_BCN20000,LARGE_OFFICIAL_ARCHIVE_OPTIONAL,200,True,True,METADATA_PROVIDER_SPLIT


In [3]:
# @title T2-I-2. Size-capped automatic acquisition and manual-inbox receipts
ISIC_ASSETS={
    "ISIC_MILK10K":[
        ("MILK10k_Training_Input.zip","https://isic-archive.s3.amazonaws.com/challenges/milk10k/MILK10k_Training_Input.zip","image_archive"),
        ("MILK10k_Training_Metadata.csv","https://isic-archive.s3.amazonaws.com/challenges/milk10k/MILK10k_Training_Metadata.csv","metadata"),
        ("MILK10k_Training_Supplement.csv","https://isic-archive.s3.amazonaws.com/challenges/milk10k/MILK10k_Training_Supplement.csv","supplement"),
        ("MILK10k_Training_GroundTruth.csv","https://isic-archive.s3.amazonaws.com/challenges/milk10k/MILK10k_Training_GroundTruth.csv","labels"),
    ],
    "ISIC_SLICE3D_PERMISSIVE":[
        ("ISIC_2024_Permissive_Training_Input.zip","https://isic-archive.s3.amazonaws.com/challenges/2024/ISIC_2024_Permissive_Training_Input.zip","image_archive"),
        ("ISIC_2024_Permissive_Training_Supplement.csv","https://isic-archive.s3.amazonaws.com/challenges/2024/ISIC_2024_Permissive_Training_Supplement.csv","metadata"),
        ("ISIC_2024_Permissive_Training_GroundTruth.csv","https://isic-archive.s3.amazonaws.com/challenges/2024/ISIC_2024_Permissive_Training_GroundTruth.csv","labels"),
    ],
}

def mendeley_zip_url(dataset_id,version):
    return f"https://api.data.mendeley.com/datasets/{dataset_id}/zip/file_downloaded?version={version}"

def discover_tcia_links(page_url):
    found=[]
    try:
        response=session.get(page_url,timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        soup=BeautifulSoup(response.text,"html.parser")
        for a in soup.find_all("a",href=True):
            text=" ".join(a.get_text(" ",strip=True).split()).lower()
            href=urljoin(response.url,a["href"])
            if ("download" in text or any(ext in href.lower() for ext in [".zip",".xlsx",".csv"])) and href.startswith("http"):
                found.append(href)
    except Exception:
        pass
    # stable dedup preserving order
    return list(dict.fromkeys(found))

def content_length_and_type(url):
    try:
        r=session.get(url,timeout=REQUEST_TIMEOUT,allow_redirects=True,stream=True,headers={"Range":"bytes=0-0"})
        size=int(r.headers.get("content-range","").split("/")[-1]) if "/" in r.headers.get("content-range","") else int(r.headers.get("content-length","0") or 0)
        info=(r.status_code,str(r.url),size,r.headers.get("content-type",""))
        r.close()
        return info
    except Exception as exc:
        return (None,url,0,f"ERROR:{type(exc).__name__}:{exc}")

def download_stream(url,destination,max_bytes):
    destination=Path(destination); destination.parent.mkdir(parents=True,exist_ok=True)
    if destination.exists() and destination.stat().st_size>0:
        return {"status":"ALREADY_PRESENT","final_url":url,"bytes":destination.stat().st_size,"sha256":sha_file(destination),"error":""}
    temp=destination.with_suffix(destination.suffix+".part")
    try:
        with session.get(url,timeout=REQUEST_TIMEOUT,allow_redirects=True,stream=True) as r:
            r.raise_for_status()
            ctype=r.headers.get("content-type","").lower()
            if "text/html" in ctype and destination.suffix.lower() not in {".html",".htm"}:
                raise RuntimeError(f"Unexpected HTML instead of asset: {r.url}")
            total=0
            h=hashlib.sha256()
            with temp.open("wb") as f:
                for chunk in r.iter_content(1024*1024):
                    if not chunk: continue
                    total+=len(chunk)
                    if total>max_bytes: raise RuntimeError("Per-file size cap exceeded during transfer")
                    f.write(chunk); h.update(chunk)
            temp.replace(destination)
            return {"status":"DOWNLOADED","final_url":str(r.url),"bytes":total,"sha256":h.hexdigest(),"error":""}
    except Exception as exc:
        if temp.exists(): temp.unlink()
        return {"status":"HOLD_DOWNLOAD_FAILED","final_url":url,"bytes":0,"sha256":"","error":f"{type(exc).__name__}: {exc}"}

asset_plan=[]
for _,row in registry.iterrows():
    dataset_id=row["dataset_id"]
    if not bool(row["auto_acquire"]) or row["expansion_role"] in {"EXCLUDE_DUPLICATE","FUTURE_MODALITY_SEED"}:
        continue
    if dataset_id in ISIC_ASSETS:
        for filename,url,kind in ISIC_ASSETS[dataset_id]:
            asset_plan.append({"dataset_id":dataset_id,"filename":filename,"url":url,"kind":kind})
    elif str(row["access_mode"])=="AUTO_MENDELEY_PUBLIC_ZIP":
        asset_plan.append({
            "dataset_id":dataset_id,
            "filename":f"{dataset_id}_official_mendeley_v{row['version']}.zip",
            "url":mendeley_zip_url(str(row["mendeley_id"]),str(row["version"])),
            "kind":"dataset_archive",
        })
    elif dataset_id=="BREAST_LESIONS_USG":
        links=discover_tcia_links(str(row["official_url"]))
        for i,url in enumerate(links):
            parsed=Path(urlparse(url).path)
            suffix=parsed.suffix.lower()
            if suffix in {".zip",".xlsx",".csv"} or "tcia-downloads" in url:
                filename=parsed.name or f"tcia_asset_{i}"
                asset_plan.append({"dataset_id":dataset_id,"filename":filename,"url":url,"kind":"official_asset"})

# If TCIA page markup exposes no stable direct URL, preserve a manual official-route hold.
if not any(x["dataset_id"]=="BREAST_LESIONS_USG" for x in asset_plan):
    asset_plan.append({
        "dataset_id":"BREAST_LESIONS_USG","filename":"",
        "url":"https://www.cancerimagingarchive.net/collection/breast-lesions-usg/",
        "kind":"MANUAL_OFFICIAL_PAGE_DOWNLOAD",
    })

plan=pd.DataFrame(asset_plan)
write_csv_once(P1/"StageT2-I_Automatic_Acquisition_Plan_v0.1.csv",plan)

total_cap=int(MAX_TOTAL_AUTO_DOWNLOAD_GB*1024**3)
single_cap=int(MAX_SINGLE_AUTO_DOWNLOAD_GB*1024**3)
used=0
receipt_rows=[]

for _,asset in plan.iterrows():
    dataset_id=asset["dataset_id"]
    inbox=ACQ_ROOT/dataset_id/"00_Raw_Inbox"
    inbox.mkdir(parents=True,exist_ok=True)
    if asset["kind"]=="MANUAL_OFFICIAL_PAGE_DOWNLOAD":
        receipt_rows.append({
            **asset.to_dict(),"status":"HOLD_MANUAL_OFFICIAL_DOWNLOAD",
            "final_url":asset["url"],"bytes":0,"sha256":"","error":"No stable direct asset URL discovered from official page."
        })
        continue
    status,final_url,reported_size,ctype=content_length_and_type(asset["url"])
    if not AUTO_DOWNLOAD_OPEN_ASSETS:
        receipt_rows.append({**asset.to_dict(),"status":"PLANNED_NOT_EXECUTED","final_url":final_url,"bytes":0,"sha256":"","error":""})
        continue
    if reported_size and reported_size>single_cap:
        receipt_rows.append({**asset.to_dict(),"status":"HOLD_SINGLE_FILE_SIZE_CAP","final_url":final_url,"bytes":reported_size,"sha256":"","error":""})
        continue
    if reported_size and used+reported_size>total_cap:
        receipt_rows.append({**asset.to_dict(),"status":"HOLD_TOTAL_SIZE_CAP","final_url":final_url,"bytes":reported_size,"sha256":"","error":""})
        continue
    destination=inbox/asset["filename"]
    result=download_stream(asset["url"],destination,single_cap)
    used+=int(result["bytes"])
    receipt_rows.append({**asset.to_dict(),**result})

# Hash any user-provided manual official files already placed in raw inboxes.
planned_paths={(r["dataset_id"],r["filename"]) for r in receipt_rows if r.get("filename")}
for _,row in registry.iterrows():
    inbox=ACQ_ROOT/row["dataset_id"]/"00_Raw_Inbox"
    inbox.mkdir(parents=True,exist_ok=True)
    for path in sorted(inbox.glob("*")):
        if not path.is_file() or path.name.endswith(".part"): continue
        if (row["dataset_id"],path.name) in planned_paths: continue
        receipt_rows.append({
            "dataset_id":row["dataset_id"],"filename":path.name,"url":"MANUAL_OFFICIAL_INBOX",
            "kind":"manual_asset","status":"MANUAL_FILE_PRESENT",
            "final_url":"","bytes":path.stat().st_size,"sha256":sha_file(path),"error":"",
        })

receipts=pd.DataFrame(receipt_rows)
write_csv_once(P1/"StageT2-I_Acquisition_Receipts_v0.1.csv",receipts)
display(receipts[["dataset_id","filename","status","bytes","error"]])
print("Automatic bytes acquired:",used)


,dataset_id,filename,status,bytes,error
0,ISIC_MILK10K,MILK10k_Training_Input.zip,DOWNLOADED,329178994,
1,ISIC_MILK10K,MILK10k_Training_Metadata.csv,DOWNLOADED,2419289,
2,ISIC_MILK10K,MILK10k_Training_Supplement.csv,DOWNLOADED,561469,
3,ISIC_MILK10K,MILK10k_Training_GroundTruth.csv,DOWNLOADED,293506,
4,BREAST_LESIONS_USG,BrEaST-Lesions_USG-images_and_masks-Dec-15-202...,DOWNLOADED,69862076,
5,BREAST_LESIONS_USG,BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx,DOWNLOADED,40173,
6,BUS_UCLM_V3,BUS_UCLM_V3_official_mendeley_v3.zip,HOLD_DOWNLOAD_FAILED,0,HTTPError: 403 Client Error: Forbidden for url...
7,HISBREAST_V2,HISBREAST_V2_official_mendeley_v2.zip,HOLD_DOWNLOAD_FAILED,0,HTTPError: 403 Client Error: Forbidden for url...


Automatic bytes acquired: 402355507


In [4]:
# @title T2-I-3. Safe extraction, file inventory and label/grouping schema audit
def safe_extract_zip(path,destination):
    destination=Path(destination); destination.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(path) as zf:
        base=destination.resolve()
        for member in zf.infolist():
            target=(destination/member.filename).resolve()
            if not str(target).startswith(str(base)+os.sep) and target!=base:
                raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
        zf.extractall(destination)

def safe_extract_tar(path,destination):
    destination=Path(destination); destination.mkdir(parents=True,exist_ok=True)
    with tarfile.open(path) as tf:
        base=destination.resolve()
        for member in tf.getmembers():
            target=(destination/member.name).resolve()
            if not str(target).startswith(str(base)+os.sep) and target!=base:
                raise RuntimeError(f"Unsafe TAR member: {member.name}")
        tf.extractall(destination)

extract_rows=[]
if AUTO_EXTRACT_ARCHIVES and len(receipts):
    for _,r in receipts.iterrows():
        if r["status"] not in {"DOWNLOADED","ALREADY_PRESENT","MANUAL_FILE_PRESENT"}: continue
        source=ACQ_ROOT/r["dataset_id"]/"00_Raw_Inbox"/r["filename"]
        if not source.is_file(): continue
        size=source.stat().st_size
        if size>int(MAX_AUTO_EXTRACT_GB*1024**3):
            extract_rows.append({"dataset_id":r["dataset_id"],"archive":r["filename"],"status":"HOLD_EXTRACT_SIZE_CAP","error":""})
            continue
        suffixes="".join(source.suffixes).lower()
        destination=ACQ_ROOT/r["dataset_id"]/"01_Extracted"
        try:
            if suffixes.endswith(".zip"):
                safe_extract_zip(source,destination)
                status="EXTRACTED"
            elif suffixes.endswith((".tar.gz",".tgz",".tar")):
                safe_extract_tar(source,destination)
                status="EXTRACTED"
            else:
                status="NOT_ARCHIVE"
            extract_rows.append({"dataset_id":r["dataset_id"],"archive":r["filename"],"status":status,"error":""})
        except Exception as exc:
            extract_rows.append({"dataset_id":r["dataset_id"],"archive":r["filename"],"status":"HOLD_UNSAFE_OR_FAILED_EXTRACTION","error":f"{type(exc).__name__}: {exc}"})
extracts=pd.DataFrame(extract_rows)
write_csv_once(P2/"StageT2-I_Extraction_Audit_v0.1.csv",extracts)

inventory_rows=[]
schema_rows=[]
image_ext={".jpg",".jpeg",".png",".bmp",".tif",".tiff",".dcm"}
for _,row in registry.iterrows():
    dataset_id=row["dataset_id"]
    dataset_root=ACQ_ROOT/dataset_id
    files=[p for p in dataset_root.rglob("*") if p.is_file() and not p.name.endswith(".part")]
    for path in files:
        inventory_rows.append({
            "dataset_id":dataset_id,
            "relative_path":str(path.relative_to(dataset_root)),
            "size_bytes":path.stat().st_size,
            "suffix":path.suffix.lower(),
            "is_image":path.suffix.lower() in image_ext,
            "sha256":sha_file(path) if path.stat().st_size<=200*1024**2 else "",
            "sha256_status":"COMPUTED" if path.stat().st_size<=200*1024**2 else "DEFERRED_LARGE_FILE",
        })
        if path.suffix.lower()==".csv" and path.stat().st_size<=100*1024**2:
            try:
                frame=pd.read_csv(path,nrows=5)
                cols=[str(c) for c in frame.columns]
                lower=[c.lower() for c in cols]
                group_candidates=[c for c in cols if any(k in c.lower() for k in ["patient","lesion","subject","case","exam"])]
                label_candidates=[c for c in cols if any(k in c.lower() for k in ["target","label","diagnos","malignan","melanoma","grade","dr"])]
                schema_rows.append({
                    "dataset_id":dataset_id,"relative_path":str(path.relative_to(dataset_root)),
                    "columns_json":json.dumps(cols,ensure_ascii=False),
                    "group_candidate_columns_json":json.dumps(group_candidates,ensure_ascii=False),
                    "label_candidate_columns_json":json.dumps(label_candidates,ensure_ascii=False),
                    "rows_read_for_schema":len(frame),"outcome_counts_computed":False,"error":"",
                })
            except Exception as exc:
                schema_rows.append({
                    "dataset_id":dataset_id,"relative_path":str(path.relative_to(dataset_root)),
                    "columns_json":"[]","group_candidate_columns_json":"[]","label_candidate_columns_json":"[]",
                    "rows_read_for_schema":0,"outcome_counts_computed":False,
                    "error":f"{type(exc).__name__}: {exc}",
                })

inventory=pd.DataFrame(inventory_rows)
schemas=pd.DataFrame(schema_rows)
write_csv_once(P2/"StageT2-I_File_Inventory_v0.1.csv",inventory)
write_csv_once(P2/"StageT2-I_Label_And_Grouping_Schema_Audit_v0.1.csv",schemas)

print("Inventoried files:",len(inventory))
print("Schema-audited CSV files:",len(schemas))
if len(schemas): display(schemas)


Inventoried files: 11010
Schema-audited CSV files: 3


,dataset_id,relative_path,columns_json,group_candidate_columns_json,label_candidate_columns_json,rows_read_for_schema,outcome_counts_computed,error
0,ISIC_MILK10K,00_Raw_Inbox/MILK10k_Training_Metadata.csv,"[""lesion_id"", ""image_type"", ""isic_id"", ""attrib...","[""lesion_id""]","[""MONET_gel_water_drop_fluid_dermoscopy_liquid""]",5,False,
1,ISIC_MILK10K,00_Raw_Inbox/MILK10k_Training_Supplement.csv,"[""isic_id"", ""diagnosis_full"", ""diagnosis_confi...",[],"[""diagnosis_full"", ""diagnosis_confirm_type""]",5,False,
2,ISIC_MILK10K,00_Raw_Inbox/MILK10k_Training_GroundTruth.csv,"[""lesion_id"", ""AKIEC"", ""BCC"", ""BEN_OTH"", ""BKL""...","[""lesion_id""]",[],5,False,


In [5]:
# @title T2-I-4. Expansion readiness, gates, decision and sealed completion
success_states={"DOWNLOADED","ALREADY_PRESENT","MANUAL_FILE_PRESENT"}
acquired_ids=set(receipts.loc[receipts["status"].isin(success_states),"dataset_id"]) if len(receipts) else set()
reachable_ids=set(preflight.loc[preflight["reachable"],"dataset_id"])
unsafe_ids=set(extracts.loc[extracts["status"]=="HOLD_UNSAFE_OR_FAILED_EXTRACTION","dataset_id"]) if len(extracts) else set()

readiness_rows=[]
for _,row in registry.iterrows():
    dataset_id=row["dataset_id"]
    role=row["expansion_role"]
    acquired=dataset_id in acquired_ids
    grouping_planned=not any(x in str(row["grouping_status"]) for x in ["NO_RELIABLE","VERIFY_PATIENT","VERIFY_"])
    if role=="EXCLUDE_DUPLICATE":
        status="EXCLUDED_DUPLICATE"
    elif dataset_id in unsafe_ids:
        status="HOLD_UNSAFE_ARCHIVE"
    elif acquired and bool(row["source_axis_compatible"]) and grouping_planned:
        status="ACQUIRED_READY_FOR_DEDUP_AND_ENDPOINT_HARMONISATION"
    elif acquired:
        status="ACQUIRED_CONDITIONAL_SCHEMA_OR_SOURCE_AXIS"
    elif row["access_mode"] in {"CREDENTIALED_DUA","MANUAL_OFFICIAL_AGREEMENT","SIGNED_NONCOMMERCIAL_RESEARCH_AGREEMENT","GRAND_CHALLENGE_REGISTRATION","QUICK_REGISTRATION_MANUAL","AUTHOR_REQUEST"}:
        status="HOLD_LEGITIMATE_MANUAL_ACCESS_REQUIRED"
    elif role.startswith("CONDITIONAL") or role.startswith("HOLD"):
        status="HOLD_CONDITIONAL_READINESS"
    elif role=="FUTURE_MODALITY_SEED":
        status="HOLD_NEW_SOURCE_AXIS_REQUIRED"
    else:
        status="HOLD_ACQUISITION_NOT_COMPLETED"
    readiness_rows.append({
        "dataset_id":dataset_id,"modality":row["modality"],"expansion_role":role,
        "source_axis_compatible":bool(row["source_axis_compatible"]),
        "official_route_reachable":dataset_id in reachable_ids,
        "asset_acquired":acquired,
        "grouping_planned":grouping_planned,
        "overlap_risk":row["overlap_risk"],
        "readiness_status":status,
        "may_contribute_to_target_level_n":status=="ACQUIRED_READY_FOR_DEDUP_AND_ENDPOINT_HARMONISATION",
        "target_outcome_scored":False,
    })
readiness=pd.DataFrame(readiness_rows)
write_csv_once(P3/"StageT2-I_Expansion_Readiness_v0.1.csv",readiness)

non_excluded=registry[registry["expansion_role"]!="EXCLUDE_DUPLICATE"]
same_axis=non_excluded[
    non_excluded["source_axis_compatible"].astype(bool)
    & ~non_excluded["expansion_role"].isin(["HOLD_GROUPING"])
]
documented=int(preflight["governance_route_documented"].sum())
reachable=int(preflight["reachable"].sum())
acquired_count=int(len(acquired_ids))
safe_extraction=not (len(extracts) and (extracts["status"]=="HOLD_UNSAFE_OR_FAILED_EXTRACTION").any())

gates=pd.DataFrame([
    {"gate":"G1_parent_integrity","passed":True,"observed":"T2-H/T3-PF self-hashes exact"},
    {"gate":"G2_registry_exact","passed":sha_file(REGISTRY_PATH)==EXPECTED_SHA["registry"],"observed":EXPECTED_SHA["registry"]},
    {"gate":"G3_blind_exclusion","passed":LOCKED_BLIND_NAMES.isdisjoint(upper_ids),"observed":"3 sentinel targets and aliases absent"},
    {"gate":"G4_expansion_capacity","passed":len(non_excluded)>=20 and len(same_axis)>=16,"observed":f"{len(non_excluded)} non-excluded / {len(same_axis)} same-axis candidates"},
    {"gate":"G5_governance_routes","passed":documented==len(registry),"observed":f"{documented}/{len(registry)} documented"},
    {"gate":"G6_official_route_reachability","passed":reachable>=16,"observed":f"{reachable}/{len(registry)} reachable or credential-gated"},
    {"gate":"G7_grouping_plan","passed":int(readiness["grouping_planned"].sum())>=18,"observed":int(readiness["grouping_planned"].sum())},
    {"gate":"G8_open_asset_acquisition","passed":acquired_count>=1,"observed":f"{acquired_count} datasets with files present"},
    {"gate":"G9_safe_extraction","passed":safe_extraction,"observed":"no unsafe archive extracted"},
    {"gate":"G10_duplicate_firewall","passed":(registry["expansion_role"]=="EXCLUDE_DUPLICATE").sum()>=1,"observed":"Rodrigues derivative explicitly excluded"},
    {"gate":"G11_no_outcome_scoring","passed":not readiness["target_outcome_scored"].any(),"observed":"schemas only; no class counts/AUC"},
    {"gate":"G12_stage12_false","passed":t3pf["stage12_authorised"] is False,"observed":t3pf["stage12_authorised"]},
])

integrity=bool(gates.loc[gates["gate"].isin([
    "G1_parent_integrity","G2_registry_exact","G3_blind_exclusion",
    "G4_expansion_capacity","G5_governance_routes","G9_safe_extraction",
    "G10_duplicate_firewall","G11_no_outcome_scoring","G12_stage12_false"
]),"passed"].all())

if not integrity:
    decision="TERMINATE_T2I_REGISTRY_OR_FIREWALL_INTEGRITY_FAILURE"
elif acquired_count>=1:
    decision="SEAL_EXPANDED_TARGET_REGISTRY_AUTHORISE_DEVELOPMENT_HARMONISATION_AND_DEDUP_ONLY"
else:
    decision="SEAL_EXPANDED_TARGET_REGISTRY_HOLD_AUTOMATIC_ACQUISITION_CONTINUE_OFFICIAL_MANUAL_ACCESS"

write_csv_once(P3/"StageT2-I_Frozen_Gates_v0.1.csv",gates)
display(readiness)
display(gates)

completion=seal(P4/"StageT2-I_Complete_v0.1.json",{
    "stage":"StageT2-I","decision":decision,
    "parent_t2h_final_record_sha256":EXPECTED_T2H_RECORD,
    "parent_t3pf_activation_record_sha256":EXPECTED_T3PF_RECORD,
    "protocol_seal_sha256":protocol["protocol_seal_sha256"],
    "registry_rows":int(len(registry)),
    "non_excluded_candidates":int(len(non_excluded)),
    "same_axis_candidates":int(len(same_axis)),
    "primary_meta_expansion_candidates":int((registry["expansion_role"]=="PRIMARY_META_EXPANSION").sum()),
    "official_routes_reachable":reachable,
    "datasets_with_assets_present":acquired_count,
    "ready_for_dedup_and_harmonisation":int((readiness["readiness_status"]=="ACQUIRED_READY_FOR_DEDUP_AND_ENDPOINT_HARMONISATION").sum()),
    "manual_or_credential_holds":int(readiness["readiness_status"].str.contains("MANUAL_ACCESS").sum()),
    "gates_passed":int(gates["passed"].sum()),"gates_total":int(len(gates)),
    "target_outcomes_scored":False,
    "method_refit_authorised":False,
    "blind_assets_touched":False,"blind_outcomes_accessed":False,
    "stage12_authorised":False,
},"final_record_sha256")

summary=f"""# Stage T2-I result summary v0.1

- Decision: `{decision}`
- Registry rows: `{len(registry)}`
- Non-excluded candidates: `{len(non_excluded)}`
- Same-source-axis candidates: `{len(same_axis)}`
- Primary meta-expansion candidates: `{int((registry['expansion_role']=='PRIMARY_META_EXPANSION').sum())}`
- Official routes reachable: `{reachable}`
- Datasets with assets present: `{acquired_count}`
- Ready for dedup/harmonisation: `{int((readiness['readiness_status']=='ACQUIRED_READY_FOR_DEDUP_AND_ENDPOINT_HARMONISATION').sum())}`
- Gates: `{int(gates['passed'].sum())}/{len(gates)}`
- Target outcomes scored: `False`
- Blind assets touched: `False`
- Blind outcomes accessed: `False`
- Final record SHA256: `{completion['final_record_sha256']}`
"""
write_once(P4/"StageT2-I_Result_Summary_v0.1.md",summary)

print("\\n========== STAGE T2-I COMPLETE ==========")
print("Decision:",decision)
print("Non-excluded candidates:",len(non_excluded))
print("Datasets with assets present:",acquired_count)
print("Ready for dedup/harmonisation:",int((readiness["readiness_status"]=="ACQUIRED_READY_FOR_DEDUP_AND_ENDPOINT_HARMONISATION").sum()))
print("Target outcomes scored:",False)
print("Blind assets touched:",False)
print("Stage 12 authorised:",False)
print("Final record SHA256:",completion["final_record_sha256"])


,dataset_id,modality,expansion_role,source_axis_compatible,official_route_reachable,asset_acquired,grouping_planned,overlap_risk,readiness_status,may_contribute_to_target_level_n,target_outcome_scored
0,MESSIDOR2,retinal_fundus,PRIMARY_META_EXPANSION,True,True,False,True,LOW,HOLD_LEGITIMATE_MANUAL_ACCESS_REQUIRED,False,False
1,MESSIDOR_ORIGINAL,retinal_fundus,PRIMARY_META_EXPANSION,True,True,False,True,LOW,HOLD_LEGITIMATE_MANUAL_ACCESS_REQUIRED,False,False
2,BRSET_V1_0_1,retinal_fundus,PRIMARY_META_EXPANSION,True,True,False,True,LOW,HOLD_LEGITIMATE_MANUAL_ACCESS_REQUIRED,False,False
3,mBRSET_V1_0,retinal_fundus,PRIMARY_META_EXPANSION,True,True,False,True,LOW,HOLD_LEGITIMATE_MANUAL_ACCESS_REQUIRED,False,False
4,ODIR5K_DR,retinal_fundus,PRIMARY_META_EXPANSION,True,True,False,True,LOW,HOLD_LEGITIMATE_MANUAL_ACCESS_REQUIRED,False,False
5,FGADR_SEG,retinal_fundus,PRIMARY_META_EXPANSION,True,True,False,False,LOW,HOLD_LEGITIMATE_MANUAL_ACCESS_REQUIRED,False,False
6,DDR,retinal_fundus,CONDITIONAL_META_EXPANSION,True,True,False,False,LOW,HOLD_CONDITIONAL_READINESS,False,False
7,ISIC_MILK10K,dermoscopy,PRIMARY_META_EXPANSION,True,True,True,True,MEDIUM_DEDUP_REQUIRED,ACQUIRED_READY_FOR_DEDUP_AND_ENDPOINT_HARMONIS...,True,False
8,PH2,dermoscopy,PRIMARY_META_EXPANSION,True,True,False,True,LOW,HOLD_LEGITIMATE_MANUAL_ACCESS_REQUIRED,False,False
9,ISIC2019_BCN20000,dermoscopy,CONDITIONAL_PROVIDER_PARTITION,True,True,False,True,HIGH_PERCEPTUAL_DEDUP_REQUIRED,HOLD_CONDITIONAL_READINESS,False,False


,gate,passed,observed
0,G1_parent_integrity,True,T2-H/T3-PF self-hashes exact
1,G2_registry_exact,True,c6224710027a7419b9b207df35831b5a2671d8900a82b8...
2,G3_blind_exclusion,True,3 sentinel targets and aliases absent
3,G4_expansion_capacity,True,23 non-excluded / 19 same-axis candidates
4,G5_governance_routes,True,24/24 documented
5,G6_official_route_reachability,True,23/24 reachable or credential-gated
6,G7_grouping_plan,True,20
7,G8_open_asset_acquisition,True,2 datasets with files present
8,G9_safe_extraction,True,no unsafe archive extracted
9,G10_duplicate_firewall,True,Rodrigues derivative explicitly excluded


\n========== STAGE T2-I COMPLETE ==========
Decision: SEAL_EXPANDED_TARGET_REGISTRY_AUTHORISE_DEVELOPMENT_HARMONISATION_AND_DEDUP_ONLY
Non-excluded candidates: 23
Datasets with assets present: 2
Ready for dedup/harmonisation: 2
Target outcomes scored: False
Blind assets touched: False
Stage 12 authorised: False
Final record SHA256: 810f4c8380263ba4cdf0cd63e4f4621362718b72ae7890ed603f1a29332d9c05


## What the decision means

A pass authorises only development-data harmonisation and deduplication for legitimately acquired candidates. It does not turn every registry row into a usable target, does not fit a new meta-predictor, and does not touch the three locked-blind sentinel datasets.

Credentialed or agreement-gated datasets remain `HOLD` until the required action is completed. Place any manually obtained official archive unchanged in that dataset's `00_Raw_Inbox` folder and rerun the notebook to generate checksums and inventory records.
